# SEP490 GPU Service on Colab (Local ZIP Version)
Chạy notebook này để đưa mã nguồn GPU Service từ máy bạn lên Colab và lấy URL cho file `.env`.

In [ ]:
!pip install pyngrok
import os
from pyngrok import ngrok

# 1. Thay đổi AuthToken của bạn ở đây
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

if len(NGROK_AUTHTOKEN) < 20 or "YOUR_NGROK" in NGROK_AUTHTOKEN:
  raise ValueError("❌ LỖI: Token không hợp lệ. Vui lòng dán chính xác NGROK_AUTHTOKEN của bạn vào biến ở trên.")
else:
  ngrok.set_auth_token(NGROK_AUTHTOKEN)
  print("Đã cấu hình ngrok thành công.")


In [ ]:
import os

# 2. Kiểm tra và giải nén mã nguồn của bạn
if not os.path.exists('/content/gpu-service.zip'):
  raise ValueError("❌ LỖI: Không tìm thấy file gpu-service.zip. Bạn hãy nhìn sang cột menu bên trái của Colab (mục Files/Tệp), bấm nút Upload (Tải lên) và chọn file gpu-service.zip ở máy tính bạn để tải lên đây nhé!")

!unzip -q -o /content/gpu-service.zip -d /content/
print("✅ Đã giải nén mã nguồn thành công!")

# Cài đặt Unsloth (thư viện chuyên dụng)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade torchao

# Cài đặt các requirements khác của gpu-service
!pip install -r /content/gpu-service/requirements.txt


In [ ]:
# 3. Khởi chạy Flask app và phơi bày qua Ngrok
# Xóa các ngrok process cũ nếu bạn bấm chạy lại nhiều lần
try:
  ngrok.kill()
except:
  pass

public_url = ngrok.connect(5000).public_url
print("\n" + "="*80)
health_url = public_url.rstrip("/") + "/health"
print(f"-> GPU_SERVICE_URL: {public_url}")
print(f"-> Health check: {health_url}")
print("="*80 + "\n")

# Thiết lập biến môi trường cần thiết cho Flask app
os.environ["PORT"] = "5000"
os.environ["HOST"] = "0.0.0.0"

# Đảm bảo chạy đúng thư mục
%cd /content/gpu-service
!python app.py
